# Bootstrap Tabular do Projeto Cross-Layer Tx+CSI

Este notebook executa a etapa inicial da camada tabular do projeto:

- localizar arquivos locais;
- baixar do Kaggle quando necessario;
- perfilar os datasets;
- aplicar o preparo inicial para treino.

Para o preprocessamento de CSI, use o pipeline `cross_layer_csi.pipelines.csi`.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_DIR =', SRC_DIR)


PROJECT_ROOT = C:\workspace\cross-layer-csi-anomaly-detection-v2
SRC_DIR = C:\workspace\cross-layer-csi-anomaly-detection-v2\src


In [2]:
from time import perf_counter

from cross_layer_csi.pipelines.tabular import TabularBootstrapPipeline

print('Iniciando bootstrap com logs de progresso...')
started_at = perf_counter()
results = TabularBootstrapPipeline(verbose=True).run()
elapsed = perf_counter() - started_at
print(f'Bootstrap concluido em {elapsed:.1f}s')
[(result.display_name, str(result.prepared.train_path), str(result.prepared.test_path)) for result in results]


Iniciando bootstrap com logs de progresso...
[09:19:09] [bootstrap] Iniciando pipeline para 3 dataset(s): ieee_cis, sparkov, ecommerce
[09:19:09] [bootstrap] [1/3] Processando `ieee_cis`...
[09:19:09] [ieee_cis] Inicio do processamento do dataset.
[09:19:09] [ieee_cis] Verificando arquivos locais...
[09:19:09] [ieee_cis] Arquivos encontrados: sample_submission, test_identity, test_transaction, train_identity, train_transaction
[09:19:09] [ieee_cis] Carregando arquivos brutos em memoria...
[09:19:09] [ieee_cis] Lendo `train_transaction.csv`...
[09:21:44] [ieee_cis] Lendo `train_identity.csv`...
[09:21:47] [ieee_cis] Mesclando transacoes e identidades de treino...
[09:21:52] [ieee_cis] Lendo `test_transaction.csv`...
[09:23:03] [ieee_cis] Lendo `test_identity.csv`...
[09:23:05] [ieee_cis] Mesclando transacoes e identidades de teste...
[09:23:07] [ieee_cis] Carga concluida: train=590540x434, competition_test=506691x433
[09:23:07] [ieee_cis] Gerando perfil dos dados e candidatos de identif

[('IEEE-CIS Fraud Detection',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\ieee_cis\\train_prepared.parquet',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\ieee_cis\\test_prepared.parquet'),
 ('Sparkov Simulated Credit Card Transactions',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\sparkov\\train_prepared.parquet',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\sparkov\\test_prepared.parquet'),
 ('Fraud E-commerce',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\ecommerce\\train_prepared.parquet',
  'C:\\workspace\\cross-layer-csi-anomaly-detection-v2\\data\\processed\\ecommerce\\test_prepared.parquet')]

In [3]:
import pandas as pd

summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'generated' / 'dataset_summary.csv')
summary


,dataset_key,display_name,split_name,rows,columns,frauds,fraud_rate,uid_choice,train_path,test_path
0,ieee_cis,IEEE-CIS Fraud Detection,train,590540,434,20663,0.034990,uid = card1_addr1 + floor(TransactionDT/86400 ...,C:\workspace\cross-layer-csi-anomaly-detection...,C:\workspace\cross-layer-csi-anomaly-detection...
1,sparkov,Sparkov Simulated Credit Card Transactions,train,1296675,23,7506,0.005789,uid = cc_num; event_id = trans_num,C:\workspace\cross-layer-csi-anomaly-detection...,C:\workspace\cross-layer-csi-anomaly-detection...
2,sparkov,Sparkov Simulated Credit Card Transactions,test,555719,23,2145,0.003860,uid = cc_num; event_id = trans_num,C:\workspace\cross-layer-csi-anomaly-detection...,C:\workspace\cross-layer-csi-anomaly-detection...
3,ecommerce,Fraud E-commerce,full,151112,11,14151,0.093646,uid = device_id; account_id = user_id; event_i...,C:\workspace\cross-layer-csi-anomaly-detection...,C:\workspace\cross-layer-csi-anomaly-detection...


In [4]:
from IPython.display import Markdown, display

profiles_md = (PROJECT_ROOT / 'reports' / 'generated' / 'dataset_profiles.md').read_text(encoding='utf-8')
display(Markdown(profiles_md))


# Dataset Profiles

## IEEE-CIS Fraud Detection

- UID adotado: `uid = card1_addr1 + floor(TransactionDT/86400 - D1)`
- Racional: O consenso mais forte encontrado na comunidade Kaggle para o IEEE-CIS e usar um UID proxy baseado em identidade de cartao/endereco combinado com a ancora temporal derivada de D1. Essa formulacao aparece nas discussoes dos vencedores e em resumos tecnicos que espelham a solucao vencedora. O projeto tambem preserva `TransactionID` como `event_id` nativo para separar identidade da transacao e identidade do usuario.
- Preparo: A preparacao inicial replica o nucleo do pipeline vencedor amplamente citado: merge de `train_transaction` com `train_identity`, construcao do UID, normalizacao dos campos `D*` por dia relativo, derivacao de features temporais de `TransactionDT` e agregacoes por UID e `card1` para `TransactionAmt`.

### Fontes
- [Kaggle Competition Overview](https://www.kaggle.com/competitions/ieee-fraud-detection/overview)
- [Kaggle Kernel - xgb-fraud-with-magic-0-9600](https://www.kaggle.com/code/cdeotte/xgb-fraud-with-magic-0-9600)
- [Kaggle Discussion - 1st Place Solution Part 1](https://www.kaggle.com/c/ieee-fraud-detection/discussion/111284)
- [Kaggle Discussion - 1st Place Solution Part 2](https://www.kaggle.com/c/ieee-fraud-detection/discussion/111308)
- [Mirror summarizing winner UID strategy](https://github.com/zehuichen123/blog/blob/67d8286f9ca254e4a60d572b708030d4496af9d7/_posts/2019-10-13-relearning-tabular-data-competition-kaggle-ieee-fraud-detection.md)
- [Fraud Dataset Benchmark IEEE preprocessing](https://github.com/amazon-science/fraud-dataset-benchmark/blob/main/src/fdb/preprocessing.py)

### Perfil bruto: train

- Linhas: 590540 | Colunas: 434 | Fraudes: 20663 | Fraud rate: 0.034990
- Top identificadores candidatos:
  - TransactionID: nunique=590540, duplicate_ratio=0.000000, max_group_size=1
  - uid_winner_card1_addr1_d1: nunique=217735, duplicate_ratio=0.631295, max_group_size=1414
  - uid_card_pack: nunique=42999, duplicate_ratio=0.927187, max_group_size=9900
  - uid_card_pack_email: nunique=94850, duplicate_ratio=0.839384, max_group_size=4002

## Sparkov Simulated Credit Card Transactions

- UID adotado: `uid = cc_num; event_id = trans_num`
- Racional: Nao existe uma competicao Kaggle oficial para essa base com solucao vencedora canonica. Para evitar UID artificial, a estrategia adotada usa os identificadores nativos do proprio dataset: `trans_num` como identificador unico da transacao e `cc_num` como identidade recorrente do cartao/cliente. `merchant` permanece como entidade secundaria para agregacoes de risco.
- Preparo: O preparo preserva o split original `fraudTrain.csv`/`fraudTest.csv`, converte timestamps e data de nascimento, deriva idade no momento da transacao, cria distancia geografica usuario-merchant e produz agregacoes por `uid` e `merchant` sobre `amt`.

### Fontes
- [Kaggle Dataset - kartik2112/fraud-detection](https://www.kaggle.com/datasets/kartik2112/fraud-detection)
- [Fraud Dataset Benchmark README](https://github.com/amazon-science/fraud-dataset-benchmark)
- [Fraud Dataset Benchmark preprocessing](https://github.com/amazon-science/fraud-dataset-benchmark/blob/main/src/fdb/preprocessing.py)

### Perfil bruto: train

- Linhas: 1296675 | Colunas: 23 | Fraudes: 7506 | Fraud rate: 0.005789
- Top identificadores candidatos:
  - trans_num: nunique=1296675, duplicate_ratio=0.000000, max_group_size=1
  - cc_num: nunique=983, duplicate_ratio=0.999242, max_group_size=3123
  - merchant: nunique=693, duplicate_ratio=0.999466, max_group_size=4403
  - cc_num__merchant: nunique=479072, duplicate_ratio=0.630538, max_group_size=24

### Perfil bruto: test

- Linhas: 555719 | Colunas: 23 | Fraudes: 2145 | Fraud rate: 0.003860
- Top identificadores candidatos:
  - trans_num: nunique=555719, duplicate_ratio=0.000000, max_group_size=1
  - cc_num: nunique=924, duplicate_ratio=0.998337, max_group_size=1474
  - merchant: nunique=693, duplicate_ratio=0.998753, max_group_size=1859
  - cc_num__merchant: nunique=328396, duplicate_ratio=0.409061, max_group_size=12

## Fraud E-commerce

- UID adotado: `uid = device_id; account_id = user_id; event_id = deterministic row-level hash/index`
- Racional: Tambem nao ha competicao Kaggle oficial com vencedor canonico. Para esta base, o projeto evita sintetizar um usuario inexistente e separa os papeis: `device_id` vira o UID operacional principal por ser a entidade mais reutilizavel em fraude online, `user_id` e mantido como identidade de conta, e o `event_id` e criado de forma deterministica porque a fonte nao traz um transaction id nativo explicito.
- Preparo: O preparo replica o conjunto de features mais recorrente nessa base: parsing temporal, `time_since_signup`, contagens por dispositivo/IP/usuario e, quando o arquivo auxiliar estiver presente, enriquecimento por pais via faixas de IP.

### Fontes
- [Kaggle Dataset - vbinh002/fraud-ecommerce](https://www.kaggle.com/datasets/vbinh002/fraud-ecommerce)
- [Fraud Dataset Benchmark preprocessing](https://github.com/amazon-science/fraud-dataset-benchmark/blob/main/src/fdb/preprocessing.py)

### Perfil bruto: full

- Linhas: 151112 | Colunas: 11 | Fraudes: 14151 | Fraud rate: 0.093646
- Top identificadores candidatos:
  - user_id: nunique=151112, duplicate_ratio=0.000000, max_group_size=1
  - device_id: nunique=137956, duplicate_ratio=0.087061, max_group_size=20
  - ip_address: nunique=143511, duplicate_ratio=0.050300, max_group_size=20
  - user_device: nunique=151112, duplicate_ratio=0.000000, max_group_size=1
